In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
# Read the sales dataset
df = pd.read_csv("sales_data_sample.csv", encoding="latin1")

# Display the first five rows
print(df.head())

# Display the number of rows and columns
print("\nDataset Shape:", df.shape)

# Check for missing values in each column
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Select numerical columns required for clustering
features = ['QUANTITYORDERED', 'PRICEEACH', 'SALES', 'MSRP']

# Store selected columns in X
X = df[features].copy()

# Replace missing values with the median of each column
X = X.fillna(X.median())

# Create a StandardScaler object
scaler = StandardScaler()

# Scale the data so all features have similar importance
X_scaled = scaler.fit_transform(X)

# Display the scaled data
print("Scaled Data:")
print(X_scaled[:5])

In [ ]:
# Create an empty list to store WCSS values
wcss = []

# Test different numbers of clusters from 1 to 10
for i in range(1, 11):

    # Create the K-Means model
    kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)

    # Train the model on scaled data
    kmeans.fit(X_scaled)

    # Store the WCSS value
    wcss.append(kmeans.inertia_)

# Plot the Elbow graph
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')

# Add graph title and labels
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")

# Display grid and graph
plt.grid()
plt.show()

In [ ]:
# Set the number of clusters based on the Elbow graph
k = 3

# Create the K-Means clustering model
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

# Train the model and assign cluster labels
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

# Display sales and their assigned cluster
print(df[['SALES', 'KMeans_Cluster']].head())

# Display the number of records in each cluster
print("\nK-Means Cluster Counts:")
print(df['KMeans_Cluster'].value_counts().sort_index())

In [ ]:
# Create a PCA object to reduce data into two dimensions
pca = PCA(n_components=2)

# Transform scaled data into two principal components
X_pca = pca.fit_transform(X_scaled)

# Set the figure size
plt.figure(figsize=(8, 5))

# Plot data points using different colors for each cluster
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df['KMeans_Cluster'],
    cmap='viridis'
)

# Add title and axis labels
plt.title("K-Means Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

# Display the graph
plt.show()

In [ ]:
# Create the Agglomerative Clustering model
hc = AgglomerativeClustering(
    n_clusters=k,
    linkage='ward'
)

# Fit the model and assign cluster labels
df['Hierarchical_Cluster'] = hc.fit_predict(X_scaled)

# Display sales and their assigned cluster
print(df[['SALES', 'Hierarchical_Cluster']].head())

# Display the number of records in each cluster
print("\nHierarchical Cluster Counts:")
print(df['Hierarchical_Cluster'].value_counts().sort_index())

In [ ]:
# Select the first 200 records for a readable dendrogram
sample_data = X_scaled[:200]

# Calculate the hierarchical linkage matrix using Ward's method
Z = linkage(sample_data, method='ward')

# Set the figure size
plt.figure(figsize=(12, 6))

# Draw the dendrogram
dendrogram(Z)

# Add title and axis labels
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Data Points")
plt.ylabel("Euclidean Distance")

# Display the graph
plt.show()

In [ ]:
# Set the figure size
plt.figure(figsize=(8, 5))

# Plot data points using hierarchical cluster labels
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df['Hierarchical_Cluster'],
    cmap='rainbow'
)

# Add title and axis labels
plt.title("Hierarchical Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

# Display the graph
plt.show()

In [ ]:
# Display the K-Means cluster counts
print("K-Means Clustering:")
print(df['KMeans_Cluster'].value_counts().sort_index())

# Display the Hierarchical cluster counts
print("\nHierarchical Clustering:")
print(df['Hierarchical_Cluster'].value_counts().sort_index())

# Calculate average sales for each K-Means cluster
print("\nAverage Sales by K-Means Cluster:")
print(df.groupby('KMeans_Cluster')['SALES'].mean())

# Calculate average sales for each Hierarchical cluster
print("\nAverage Sales by Hierarchical Cluster:")
print(df.groupby('Hierarchical_Cluster')['SALES'].mean())